# Personal Thought Archive: Exploratory Data Analysis

Exploratory data analysis (EDA) is the process of understanding a dataset before modeling it or making conclusions. In this notebook we will inspect structure, quality, time patterns, language, post types, text length, and engagement.

**Privacy rule:** this notebook does not display tweet text. All results are aggregate counts, statistics, or redacted rankings.

## How to use this notebook

A notebook contains two main cell types:

- **Markdown cells** explain ideas in ordinary language.
- **Code cells** run Python. Select a cell and press `Shift + Enter` to execute it.

Run the cells from top to bottom because later cells use variables created earlier.

## 1. Import tools and locate the dataset

An `import` makes code from a library available in our notebook. We use:

- `json` to load the cleaned JSON file while preserving tweet IDs as strings.
- `pandas` to work with tabular data through a structure called a DataFrame.
- `matplotlib` to create charts.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 30)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "tweets_clean.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "tweets_clean.json"
if not DATA_FILE.exists():
    raise FileNotFoundError("Run src/parse_tweets.py before this notebook.")

print("Dataset located successfully.")

## 2. Load the records into a DataFrame

The JSON file contains a list of tweet records. `json.load` turns that list into Python objects. `pd.DataFrame` arranges those records into rows and columns. Each row represents one tweet, and each column represents one field.

In [ ]:
with DATA_FILE.open(encoding="utf-8") as file:
    records = json.load(file)

df = pd.DataFrame(records)
print(f"Loaded {len(df):,} records and {len(df.columns)} columns.")

## 3. Inspect shape, columns, and data types

The **shape** tells us the number of rows and columns. A column's **data type** tells pandas how to treat its values—for example, text, integers, or booleans. We display the schema rather than sample rows so private text stays hidden.

In [ ]:
schema = pd.DataFrame({
    "column": df.columns,
    "data_type": [str(dtype) for dtype in df.dtypes],
})

print(f"Dataset shape: {df.shape}")
schema

## 4. Check missing values and duplicates

Missing values are not always errors. For example, a normal standalone tweet should have no `reply_to_tweet_id`. Duplicate IDs, however, would mean the same tweet appears more than once.

In [ ]:
quality_report = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1),
})

print(f"Duplicate tweet IDs: {df['id'].duplicated().sum()}")
quality_report

## 5. Convert dates and create simple features

A **feature** is a useful characteristic derived from existing data. From each timestamp and text we derive year, month, character count, and word count. We also combine reply and retweet flags into a readable post type. These additions exist only in memory; they do not alter the JSON file.

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["year"] = df["created_at"].dt.year
df["month"] = df["created_at"].dt.strftime("%Y-%m")
df["character_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

df["post_type"] = "Original"
df.loc[df["is_reply"], "post_type"] = "Reply"
df.loc[df["is_retweet"], "post_type"] = "Retweet"

print(f"Earliest tweet: {df['created_at'].min().date()}")
print(f"Latest tweet:   {df['created_at'].max().date()}")
print("Derived features created successfully.")

## 6. Tweets per year

A frequency table counts how many records belong to each year. Remember that the first and last years may be incomplete, so lower counts do not necessarily mean your behavior changed.

In [ ]:
tweets_per_year = df.groupby("year").size()
display(tweets_per_year.rename("tweet_count").to_frame())

ax = tweets_per_year.plot(
    kind="bar", figsize=(9, 4), color="#1d9bf0", title="Tweets per year"
)
ax.set_xlabel("Year")
ax.set_ylabel("Number of tweets")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 7. Activity by month

Yearly totals can hide shorter bursts or quiet periods. Monthly counts provide a more detailed time series. This chart describes posting frequency; it does not explain why activity changed.

In [ ]:
tweets_per_month = df.groupby("month").size()

ax = tweets_per_month.plot(
    figsize=(12, 4), color="#7856ff", title="Tweets per month"
)
ax.set_xlabel("Month")
ax.set_ylabel("Number of tweets")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8. Original tweets, replies, and retweets

Separating post types prevents us from treating all records as the same kind of writing. For example, replies may be shorter because they depend on conversational context.

In [ ]:
post_type_counts = df["post_type"].value_counts()
post_type_percent = (df["post_type"].value_counts(normalize=True) * 100).round(1)
post_type_summary = pd.DataFrame({
    "count": post_type_counts,
    "percent": post_type_percent,
})
display(post_type_summary)

ax = post_type_counts.plot(
    kind="bar", figsize=(8, 4), color=["#00ba7c", "#1d9bf0", "#f91880"],
    title="Records by post type",
)
ax.set_xlabel("Post type")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 9. Language distribution

The archive's `language` field is an automated label from X. It is useful for a first overview but may occasionally misclassify short or multilingual posts. We show only the ten most common labels to keep the chart readable.

In [ ]:
top_languages = df["language"].fillna("unknown").value_counts().head(10)
display(top_languages.rename("tweet_count").to_frame())

ax = top_languages.sort_values().plot(
    kind="barh", figsize=(8, 5), color="#ff7a00", title="Top language labels"
)
ax.set_xlabel("Number of tweets")
ax.set_ylabel("Language code")
plt.tight_layout()
plt.show()

## 10. Tweet length distribution

A distribution shows which values are common and which are unusual. Character counts are especially relevant on X because posts have length limits. A long right tail means a small number of posts are much longer than most.

In [ ]:
display(df[["character_count", "word_count"]].describe().round(1))

ax = df["character_count"].plot(
    kind="hist", bins=30, figsize=(9, 4), color="#1d9bf0", edgecolor="white",
    title="Distribution of tweet character counts",
)
ax.set_xlabel("Characters")
ax.set_ylabel("Number of tweets")
plt.tight_layout()
plt.show()

## 11. Engagement overview

The median describes a typical record better than the mean when a few posts have unusually high engagement. Engagement is influenced by many factors, so it should not be treated as a direct measure of writing quality.

In [ ]:
engagement_summary = df[["likes", "retweets"]].describe().round(2)
engagement_summary

## 12. Ten most-liked original tweets (text redacted)

For privacy, this ranking displays no tweet IDs or text. It helps verify engagement values and identify dates you may want to investigate privately later.

In [ ]:
top_liked_original = (
    df.loc[df["post_type"].eq("Original"), ["created_at", "likes", "retweets"]]
    .nlargest(10, "likes")
    .reset_index(drop=True)
)
top_liked_original.index = top_liked_original.index + 1
top_liked_original["created_at"] = top_liked_original["created_at"].dt.date
top_liked_original.index.name = "rank"
top_liked_original

## 13. What we can and cannot conclude

After running the notebook, write down three observations and one limitation. Use careful language:

- Good: “The archive contains more records in 2025 than in 2023.”
- Too strong: “I became more creative in 2025.”

Counts describe what is in the archive. They do not automatically explain motivation, personality, or causation.

### Your notes

1. Observation:
2. Observation:
3. Observation:
4. Limitation: